In [ ]:
import numpy as np
import pandas as pd
from sklearn import feature_extraction, linear_model, model_selection, preprocessing

In [ ]:
train_df = pd.read_csv('/home/karen/Documents/Datasets/Disaster_tweets/train.csv')
test_df = pd.read_csv('/home/karen/Documents/Datasets/Disaster_tweets/test.csv')
submission_df = pd.read_csv('/home/karen/Documents/Datasets/Disaster_tweets/sample_submission.csv')

In [ ]:
train_df

In [ ]:
train_df.info()

In [ ]:
train_df.target.value_counts()

In [ ]:
#Example of disaster tweet
train_df[train_df['target'] == 1]['text'].values[1]

In [ ]:
#Non disaster tweet
train_df[train_df['target'] == 0]['text'].values[1]

### Vectorization using count_vectorizer

In [ ]:
#Build vectors 
from sklearn.feature_extraction.text import CountVectorizer
count_vectorizer = CountVectorizer()

#get counts for the first 5 tweets in the data
example_vectors = count_vectorizer.fit_transform(train_df['text'][:5])

In [ ]:
# we use .todense() here because these vectors are "sparse" (only non-zero elements are kept to save space)
print(example_vectors[0].todense().shape)
print(example_vectors[0].todense())

In [ ]:
train_vectors = count_vectorizer.fit_transform(train_df["text"])

# note that we're NOT using .fit_transform() here. Using just .transform() makes sure
# that the tokens in the train vectors are the only ones mapped to the test vectors - 
# i.e. that the train and test vectors use the same set of tokens.
test_vectors = count_vectorizer.transform(test_df["text"])

In [ ]:
from sklearn.linear_model import RidgeClassifier
clf = RidgeClassifier()

In [ ]:
# from sklearn.model_selection import cross_val_score
# scores = cross_val_score(clf, train_vectors, train_df["target"], cv=3, scoring="f1")
# scores

In [ ]:
# clf.fit(train_vectors, train_df["target"])

### Vectorization usinf TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

In [ ]:
tfidf_train_vectors = tfidf_vectorizer.fit_transform(train_df["text"])

tfidf_test_vectors = tfidf_vectorizer.transform(test_df["text"])

In [ ]:
clf.fit(tfidf_train_vectors, train_df["target"])

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(clf, tfidf_train_vectors, train_df["target"], cv=3, scoring="f1")
scores

### Using Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier()

classifier.fit(tfidf_train_vectors, train_df["target"])

In [ ]:
scores = cross_val_score(classifier, tfidf_train_vectors, train_df["target"], cv=3, scoring="f1")
scores

### Using XG-Boost

In [ ]:
import xgboost as xgb

xgb_cl = xgb.XGBClassifier()

In [ ]:
xgb_cl.fit(tfidf_train_vectors, train_df["target"])

In [ ]:
scores = cross_val_score(xgb_cl, tfidf_train_vectors, train_df["target"], cv=3, scoring="f1")
scores

### Using LSTM

In [ ]:
from keras.models import Model
from keras.layers import LSTM, Activation, Dense, Dropout, Input, Embedding
from keras.optimizers import RMSprop
from keras.preprocessing.text import Tokenizer
from keras.preprocessing import sequence
from keras.utils import to_categorical
from keras.callbacks import EarlyStopping

In [ ]:
max_len = 150
max_words = 1000
tok = Tokenizer(num_words=max_words)
tok.fit_on_texts(train_df["text"])
sequences = tok.texts_to_sequences(train_df["text"])
sequences_matrix = sequence.pad_sequences(sequences,maxlen=max_len)

def RNN():
    inputs = Input(name='inputs',shape=[max_len])
    layer = Embedding(max_words,50,input_length=max_len)(inputs)
    layer = LSTM(64)(layer)
    layer = Dense(256,name='FC1')(layer)
    layer = Activation('relu')(layer)
    layer = Dropout(0.5)(layer)
    layer = Dense(1,name='out_layer')(layer)
    layer = Activation('sigmoid')(layer)
    model = Model(inputs=inputs,outputs=layer)
    return model

In [ ]:
model = RNN()
model.summary()
model.compile(loss='binary_crossentropy',optimizer=RMSprop(),metrics=['accuracy'])

In [ ]:
model.fit(sequences_matrix, train_df["target"],batch_size=128,epochs=10,
          validation_split=0.2,callbacks=[EarlyStopping(monitor='val_loss',min_delta=0.0001)])

In [ ]:
test_sequences = tok.texts_to_sequences(test_df["text"])
test_sequences_matrix = sequence.pad_sequences(test_sequences,maxlen=max_len)

### Submission

In [ ]:
submission_df["target"] = clf.predict(tfidf_test_vectors)

In [ ]:
submission_df

In [ ]:
submission_df.to_csv("submission.csv", index=False)